In [0]:
#Chart for pipeline

from pyspark.sql.functions import to_date, sum, count, col

daily_metrics = spark.table("dev_catalog.pipeline_schema.pipeline_runs") \
    .withColumn("date", to_date("run_time")) \
    .groupBy("date") \
    .agg(
        count("*").alias("total_runs"),
        sum("records_processed").alias("total_records"),
        sum("bad_records").alias("total_bad_records")
    ) \
    .withColumn(
        "total_good_records",
        col("total_records") - col("total_bad_records")
    )

display(daily_metrics)

In [0]:
# Success vs failure chart

status_metrics = spark.table("dev_catalog.pipeline_schema.pipeline_runs") \
    .groupBy("status") \
    .count()

display(status_metrics)

In [0]:
#Suceess rate

from pyspark.sql.functions import when

success_rate = spark.table("dev_catalog.pipeline_schema.pipeline_runs") \
    .agg(
        (sum(when(col("status") == "SUCCESS", 1).otherwise(0)) / count("*") * 100)
        .alias("success_rate")
    )

display(success_rate)

In [0]:
#Latest runs
latest_runs = spark.table("dev_catalog.pipeline_schema.pipeline_runs") \
    .orderBy(col("run_time").desc()) \
    .limit(20)

display(latest_runs)